# Open-source LLMs przez OpenRouter — zero-shot i few-shot na LIAR

Testujemy 4 darmowe modele LLM przez OpenRouter:
- `openai/gpt-oss-20b:free`
- `google/gemma-4-31b-it:free`
- `cognitivecomputations/dolphin-mistral-24b-venice-edition:free`
- `meta-llama/llama-3.3-70b-instruct:free`

Pobieramy przygotowany dataset z Drive:

In [18]:
from google.colab import drive
import os
drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/fakenews_cache'
TOK_DIR   = os.path.join(CACHE_DIR, 'tokenized')

print('TOK_DIR:', os.listdir(TOK_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TOK_DIR: ['train_2cl.pt', 'valid_2cl.pt', 'test_2cl.pt', 'train_6cl.pt', 'valid_6cl.pt', 'test_6cl.pt', 'y_train_2cl.pt', 'y_test_2cl.pt', 'y_valid_2cl.pt', 'y_train_6cl.pt', 'y_valid_6cl.pt', 'y_test_6cl.pt', 'df_train_2cl.parquet', 'df_test_2cl.parquet', 'df_valid_2cl.parquet', 'df_train_6cl.parquet', 'df_valid_6cl.parquet', 'df_test_6cl.parquet']


In [19]:
%pip install -q requests tqdm scikit-learn pandas

In [20]:
import pandas as pd

VERSION = '2cl'

train = pd.read_parquet(os.path.join(TOK_DIR, f'df_train_{VERSION}.parquet'))
test  = pd.read_parquet(os.path.join(TOK_DIR, f'df_test_{VERSION}.parquet'))

print('train:', train.shape, ' test:', test.shape)

train: (10240, 12)  test: (1267, 12)


## Konfiguracja OpenRouter

Klucz **MUSI** byc w zmiennej srodowiskowej. Na Colabie ustawia sie przez `userdata.get('OPENROUTER_API_KEY')`.

In [21]:
import os, re, time, getpass
import requests
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

if not os.environ.get('OPENROUTER_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    except Exception:
        os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OPENROUTER_API_KEY: ')

OPENROUTER_API_KEY = os.environ['OPENROUTER_API_KEY']
OPENROUTER_URL = 'https://openrouter.ai/api/v1/chat/completions'

MODELS = [
    'openai/gpt-oss-20b:free',
    'google/gemma-4-31b-it:free',
    'cognitivecomputations/dolphin-mistral-24b-venice-edition:free',
    'meta-llama/llama-3.3-70b-instruct:free',
]

MAX_TEST_SAMPLES = 120
TEMPERATURE = 0
MAX_TOKENS = 64 #zamienione z 8 zeby nie bylo unparsed
SLEEP_BETWEEN_CALLS = 0.15

LABEL_INT_TO_STR = {0: 'fake', 1: 'true'}
LABEL_STR_TO_INT = {'fake': 0, 'true': 1}

SYSTEM_PROMPT = (
    'You are a strict binary classifier for misinformation detection. '
    'Return ONLY one token: fake or true.'
)

few_shot_examples = []
for label_int in [0, 1, 0, 1]:
    row = train[train['label'] == label_int].sample(
        1, random_state=42 + len(few_shot_examples)).iloc[0]
    few_shot_examples.append({'statement': row['statement'],
                              'label': LABEL_INT_TO_STR[label_int]})

few_shot_block = '\n'.join([
    f"statement: {e['statement']}\nlabel: {e['label']}" for e in few_shot_examples
])

print('Few-shot examples prepared:', len(few_shot_examples))

Few-shot examples prepared: 4


In [22]:
def build_user_prompt(statement, mode='zero'):
    if mode == 'zero':
        return (
            'Task: classify if the statement is fake or true.\n'
            'Return only: fake OR true.\n\n'
            f'statement: {statement}'
        )
    return (
        'Task: classify if the statement is fake or true.\n'
        'Use examples below.\n'
        'Return only: fake OR true.\n\n'
        f'{few_shot_block}\n\n'
        f'statement: {statement}\n'
        'label:'
    )


def parse_label(text):
    t = (text or '').strip().lower()
    m = re.search(r'\b(fake|true)\b', t)
    if not m:
        return None
    return m.group(1)


def call_openrouter(model, user_prompt, retries=3):
    headers = {
        'Authorization': f'Bearer {OPENROUTER_API_KEY}',
        'Content-Type': 'application/json',
    }
    payload = {
        'model': model,
        'temperature': TEMPERATURE,
        'max_tokens': MAX_TOKENS,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
    }
    for attempt in range(retries):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers,
                                 json=payload, timeout=45)
            if resp.status_code == 200:
                data = resp.json()
                if 'choices' in data and data['choices']:
                    return data['choices'][0]['message'].get('content')
                err = data.get('error', {}).get('message', 'no_choices')
                return f'ERROR_BODY_{str(err)[:80]}'
            if resp.status_code in [429, 500, 502, 503, 504] and attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
                continue
            return f'ERROR_STATUS_{resp.status_code}'
        except requests.RequestException:
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
                continue
            return 'ERROR_REQUEST'


def evaluate_model_llm(model, mode='zero', n_samples=MAX_TEST_SAMPLES):
    df = test.sample(n=min(n_samples, len(test)), random_state=42).reset_index(drop=True)
    y_true, y_pred = [], []
    invalid = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'{model} | {mode}'):
        prompt = build_user_prompt(row['statement'], mode=mode)
        raw = call_openrouter(model, prompt)
        lbl = parse_label(raw)
        if lbl is None:
            invalid += 1
            lbl = 'fake'
        y_true.append(row['label'])
        y_pred.append(LABEL_STR_TO_INT[lbl])
        time.sleep(SLEEP_BETWEEN_CALLS)

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred,
                                                   average='macro', zero_division=0)
    print(f'\n=== {model} | {mode} ===  acc={acc:.3f}  macro-F1={f1:.3f}  unparsed={invalid}')
    print(classification_report(y_true, y_pred, zero_division=0))
    return {
        'model': model, 'mode': mode, 'samples': len(df),
        'invalid_outputs': invalid,
        'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1,
    }

In [23]:
llm_results = []
for model in MODELS:
    llm_results.append(evaluate_model_llm(model, mode='zero',
                                           n_samples=MAX_TEST_SAMPLES))
    llm_results.append(evaluate_model_llm(model, mode='few',
                                           n_samples=MAX_TEST_SAMPLES))

llm_results_df = pd.DataFrame(llm_results)
llm_results_df

openai/gpt-oss-20b:free | zero:   0%|          | 0/120 [00:00<?, ?it/s]


=== openai/gpt-oss-20b:free | zero ===  acc=0.467  macro-F1=0.440  unparsed=0
              precision    recall  f1-score   support

           0       0.44      0.77      0.56        53
           1       0.56      0.22      0.32        67

    accuracy                           0.47       120
   macro avg       0.50      0.50      0.44       120
weighted avg       0.50      0.47      0.43       120



openai/gpt-oss-20b:free | few:   0%|          | 0/120 [00:00<?, ?it/s]


=== openai/gpt-oss-20b:free | few ===  acc=0.492  macro-F1=0.438  unparsed=0
              precision    recall  f1-score   support

           0       0.46      0.91      0.61        53
           1       0.69      0.16      0.27        67

    accuracy                           0.49       120
   macro avg       0.57      0.53      0.44       120
weighted avg       0.59      0.49      0.42       120



google/gemma-4-31b-it:free | zero:   0%|          | 0/120 [00:00<?, ?it/s]


=== google/gemma-4-31b-it:free | zero ===  acc=0.442  macro-F1=0.306  unparsed=120
              precision    recall  f1-score   support

           0       0.44      1.00      0.61        53
           1       0.00      0.00      0.00        67

    accuracy                           0.44       120
   macro avg       0.22      0.50      0.31       120
weighted avg       0.20      0.44      0.27       120



google/gemma-4-31b-it:free | few:   0%|          | 0/120 [00:00<?, ?it/s]

KeyboardInterrupt: 

Zatrzymałam drugi run bo się nie parsowało chyba przez zbyt dużą liczbę requestów :(( Poniżej wyniki z pierwszego runu

In [13]:
ranking_llm = (llm_results_df
               .sort_values(['f1', 'accuracy'], ascending=False)
               .reset_index(drop=True)
               .round(4))
print('Ranking (LLM zero-shot / few-shot):')
ranking_llm

Ranking (LLM zero-shot / few-shot):


,model,mode,samples,invalid_outputs,accuracy,precision,recall,f1
0,openai/gpt-oss-20b:free,zero,120,0,0.5000,0.5460,0.5325,0.4754
1,openai/gpt-oss-20b:free,few,120,0,0.5000,0.5580,0.5365,0.4667
2,google/gemma-4-31b-it:free,zero,120,120,0.4417,0.2208,0.5000,0.3064
3,google/gemma-4-31b-it:free,few,120,120,0.4417,0.2208,0.5000,0.3064
4,cognitivecomputations/dolphin-mistral-24b-veni...,zero,120,120,0.4417,0.2208,0.5000,0.3064
5,cognitivecomputations/dolphin-mistral-24b-veni...,few,120,120,0.4417,0.2208,0.5000,0.3064
6,meta-llama/llama-3.3-70b-instruct:free,zero,120,120,0.4417,0.2208,0.5000,0.3064
7,meta-llama/llama-3.3-70b-instruct:free,few,120,120,0.4417,0.2208,0.5000,0.3064


In [14]:
import os
SAVE_DIR = '/content/drive/MyDrive/fakenews_cache/results'
os.makedirs(SAVE_DIR, exist_ok=True)
llm_results_df.to_csv(f'{SAVE_DIR}/openrouter_free_results.csv', index=False)
print('zapisano:', f'{SAVE_DIR}/openrouter_free_results.csv')


zapisano: /content/drive/MyDrive/fakenews_cache/results/openrouter_free_results.csv
